# Lab: Image Classification with CNN (TensorFlow Version)

**Converted from PyTorch → TensorFlow/Keras**

**Key differences vs PyTorch original:**
- `ResNet18` → `ResNet50` (TensorFlow has no ResNet18; ResNet50 is the standard equivalent available via `tf.keras.applications`)
- `ImageFolder` → `image_dataset_from_directory` (Keras built-in, same behavior)
- `CyclicLR` → custom `CyclicLRCallback` (TF has no built-in; we implement triangular2 mode manually)
- `model.fc` (512) → `Dense` head after `GlobalAveragePooling2D` (ResNet50 backbone outputs 2048 features)
- `model.eval() / torch.no_grad()` → `training=False` / `model.predict()` (Keras convention)

Estimated time: **60 minutes**

## Table of Contents
1. [Install and Import Libraries](#1-install-and-import-libraries)
2. [Image Processing and Dataset Preparation](#2-image-processing-and-dataset-preparation)
3. [Load Model and Train](#3-load-model-and-train)
4. [Practice Exercise](#4-practice-exercise)

---
## 1. Install and Import Libraries

In [ ]:
# Core libraries

!pip install tqdm pillow requests

In [ ]:
# OS and data utilities
import os
import json
import shutil
import random
import zipfile
import io
import copy
import time
from datetime import datetime
import requests

# Data and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow.keras.layers as layers
from tensorflow.keras import Model
from tensorflow.keras.optimizers import SGD

# Reproducibility
tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)

print("TensorFlow version:", tf.__version__)

In [ ]:
# Detect CPU/GPU (TF handles this automatically, but we print it for clarity)
gpus = tf.config.list_physical_devices('GPU')
device_info = f"{len(gpus)} GPU(s) available" if gpus else "Running on CPU"
print("Device:", device_info)

### Helper: Plot training loss and validation accuracy

Same behavior as the PyTorch `plot_stuff()` — dual y-axis, loss on left (red), accuracy on right (blue).

In [ ]:
def plot_stuff(COST, ACC):
    """
    Plots training loss and validation accuracy on a dual y-axis figure.
    
    Parameters:
        COST (list): Average training loss per epoch.
        ACC  (list): Validation accuracy per epoch (0.0 – 1.0).
    """
    fig, ax1 = plt.subplots()

    color = 'tab:red'
    ax1.plot(COST, color=color)
    ax1.set_xlabel('Epoch', color=color)
    ax1.set_ylabel('Total Loss', color=color)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Accuracy', color=color)
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', labelcolor=color)

    fig.tight_layout()
    plt.show()

### Helper: Display a normalized image

Reverses ImageNet normalization so the image looks correct when plotted.  
In TF, images are `(H, W, C)` NumPy arrays — no `.permute()` needed.

In [ ]:
def imshow_(inp, title=None):
    """
    Displays an image tensor after reversing ImageNet normalization.

    Parameters:
        inp   : NumPy array of shape (H, W, C), normalized.
        title : Optional string title for the plot.
    """
    # inp is already (H, W, C) in TF — no permute needed
    inp = inp.numpy() if hasattr(inp, 'numpy') else inp
    print("Image shape:", inp.shape)

    # Undo ImageNet normalization
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)

    plt.imshow(inp)
    if title:
        plt.title(title)
    plt.show()

---
## 2. Image Processing and Dataset Preparation

### Download and extract the dataset

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QkkP2jRxVvxHKMPg8lnkwQ/transfer-learning-with-cnn-15-2025-06-24-t-12-35-01-829-z.zip"

response = requests.get(url)
if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        zip_ref.extractall("hotdog_nothotdog")
    print("Download and extraction complete.")
else:
    print("Failed to download file:", response.status_code)

### Split images into train / val folders (90% / 10%)

In [ ]:
source_dir       = "hotdog_nothotdog/transfer-learning-with-cnn-15-2025-06-24-t-12-35-01-829-z"
annotations_file = os.path.join(source_dir, "_annotations.json")

with open(annotations_file, "r") as f:
    annotations = json.load(f)

train_ratio = 0.9
output_dir  = "dataset"

# Build label → image list
label_to_images = {}
for filename, entry in annotations["annotations"].items():
    label = entry[0]["label"]
    label_to_images.setdefault(label, []).append(filename)

# Shuffle and copy
for label, image_list in label_to_images.items():
    random.shuffle(image_list)
    train_cutoff  = int(len(image_list) * train_ratio)
    train_images  = image_list[:train_cutoff]
    val_images    = image_list[train_cutoff:]

    for split, split_images in zip(["train", "val"], [train_images, val_images]):
        out_path = os.path.join(output_dir, split, label)
        os.makedirs(out_path, exist_ok=True)
        for img_name in split_images:
            src = os.path.join(source_dir, img_name)
            dst = os.path.join(out_path, img_name)
            shutil.copy2(src, dst)

print("Train/Val split complete.")

### Load datasets

`image_dataset_from_directory` is the direct Keras equivalent of PyTorch's `ImageFolder`.  
It reads the folder structure and assigns labels automatically.

> **Note on normalization:** PyTorch used `transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])`.  
> In TF we use `ResNet50`'s own `preprocess_input`, which applies the same ImageNet normalization (caffe-style: BGR channel swap + mean subtraction). This is the correct preprocessor for the ResNet50 weights.

In [ ]:
IMAGE_SIZE  = (224, 224)
batch_size  = 32

# Training dataset — with augmentation
train_dataset_raw = tf.keras.utils.image_dataset_from_directory(
    "dataset/train",
    image_size=IMAGE_SIZE,
    batch_size=batch_size,
    shuffle=True,
    seed=0,
    label_mode='int'
)

# Validation dataset — no augmentation, batch size 1 (same as original)
val_dataset_raw = tf.keras.utils.image_dataset_from_directory(
    "dataset/val",
    image_size=IMAGE_SIZE,
    batch_size=1,
    shuffle=False,
    seed=0,
    label_mode='int'
)

class_names = train_dataset_raw.class_names
n_classes   = len(class_names)
print("Classes:", class_names)
print("Number of classes:", n_classes)

In [ ]:
# Data augmentation layer (equivalent to RandomHorizontalFlip + RandomRotation in PyTorch)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(5 / 360),   # 5 degrees, expressed as a fraction of 2π
])

def preprocess_train(images, labels):
    images = data_augmentation(images, training=True)  # apply augmentation
    images = preprocess_input(images)                  # ResNet50 ImageNet normalization
    return images, labels

def preprocess_val(images, labels):
    images = preprocess_input(images)                  # normalization only, no augmentation
    return images, labels

# Apply preprocessing and prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset_raw.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_dataset   = val_dataset_raw.map(preprocess_val,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

n_val = sum(1 for _ in val_dataset_raw.unbatch())  # total validation samples
print(f"Validation samples: {n_val}")

### Visualize a few validation samples

In [ ]:
count = 0
for images, labels in val_dataset.unbatch().batch(1):
    img = images[0].numpy()
    imshow_(img, title=f"y = {labels[0].numpy()} ({class_names[labels[0].numpy()]})")
    count += 1
    if count == 3:
        break

---
## 3. Load Model and Train

### Hyperparameters

In [ ]:
n_epochs   = 10
lr         = 0.000001
momentum   = 0.9
use_lr_scheduler = True
base_lr    = 0.001
max_lr     = 0.01

### Cyclic LR Callback (triangular2 mode)

TensorFlow has no built-in `CyclicLR`. We implement the `triangular2` schedule from the PyTorch original:  
LR oscillates between `base_lr` and `max_lr`, halving the amplitude each cycle.

In [ ]:
class CyclicLRCallback(tf.keras.callbacks.Callback):
    """
    Implements PyTorch's CyclicLR triangular2 schedule as a Keras Callback.

    In triangular2 mode the max_lr is halved after every complete cycle.
    One cycle = 2 * step_size_up steps (here: epochs).
    """
    def __init__(self, base_lr, max_lr, step_size_up=5):
        super().__init__()
        self.base_lr     = base_lr
        self.max_lr      = max_lr
        self.step_size   = step_size_up
        self.cycle_count = 0
        self.step_count  = 0

    def on_epoch_begin(self, epoch, logs=None):
        cycle      = epoch // (2 * self.step_size)       # which cycle are we in?
        x          = epoch % (2 * self.step_size)        # position within the cycle
        scale      = 1.0 / (2 ** cycle)                  # triangular2: halve amplitude each cycle
        if x < self.step_size:                            # ascending half
            lr = self.base_lr + (self.max_lr - self.base_lr) * (x / self.step_size) * scale
        else:                                             # descending half
            lr = self.base_lr + (self.max_lr - self.base_lr) * ((2*self.step_size - x) / self.step_size) * scale
        tf.keras.backend.set_value(self.model.optimizer.learning_rate, lr)
        print(f"  [CyclicLR] Epoch {epoch+1}: lr = {lr:.6f}")

### Build the model

**Architecture mapping:**

| PyTorch | TensorFlow |
|---|---|
| `models.resnet18(pretrained=True)` | `ResNet50(weights='imagenet', include_top=False)` |
| `param.requires_grad = False` | `base_model.trainable = False` |
| `model.fc = nn.Linear(512, n_classes)` | `GlobalAveragePooling2D` → `Dense(n_classes)` |
| `nn.CrossEntropyLoss()` | `SparseCategoricalCrossentropy(from_logits=True)` |
| `torch.optim.SGD(...)` | `SGD(learning_rate=lr, momentum=momentum)` |

In [ ]:
# Load pretrained ResNet50 without its top classification layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all base model weights — only train the new head
base_model.trainable = False

# Build the full model: base → GlobalAveragePooling → Dense output
inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)   # training=False: keeps BN layers frozen
x       = layers.GlobalAveragePooling2D()(x)   # (batch, 7, 7, 2048) → (batch, 2048)
outputs = layers.Dense(n_classes)(x)           # (batch, 2048) → (batch, n_classes), logits

model = Model(inputs, outputs)
model.summary()

In [ ]:
# Compile the model
# SparseCategoricalCrossentropy(from_logits=True) == nn.CrossEntropyLoss() in PyTorch
optimizer = SGD(learning_rate=lr, momentum=momentum)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

### Train the model

The `train_model()` function mirrors the PyTorch version step-by-step:  
- Tracks train loss and validation accuracy per epoch  
- Saves the best model weights (by val accuracy)  
- Applies the cyclic LR scheduler each epoch  

In [ ]:
def train_model(model, train_loader, val_loader, n_epochs, n_val, print_=True):
    """
    Custom training loop — mirrors the PyTorch train_model() function.

    Returns:
        accuracy_list : list of validation accuracy per epoch
        loss_list     : list of mean training loss per epoch
        model         : model loaded with best weights
    """
    loss_fn       = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    loss_list     = []
    accuracy_list = []
    best_accuracy = 0.0
    best_weights  = model.get_weights()  # equivalent to copy.deepcopy(model.state_dict())

    print("The first epoch may take several minutes...")

    for epoch in tqdm(range(n_epochs)):
        # --- Learning rate update (cyclic) ---
        if use_lr_scheduler:
            cycle    = epoch // (2 * 5)
            x_pos    = epoch % (2 * 5)
            scale    = 1.0 / (2 ** cycle)
            if x_pos < 5:
                new_lr = base_lr + (max_lr - base_lr) * (x_pos / 5) * scale
            else:
                new_lr = base_lr + (max_lr - base_lr) * ((10 - x_pos) / 5) * scale
            model.optimizer.learning_rate.assign(new_lr)

        # --- Training phase ---
        epoch_losses = []
        for x_batch, y_batch in train_loader:
            with tf.GradientTape() as tape:
                logits = model(x_batch, training=True)          # forward pass
                loss   = loss_fn(y_batch, logits)               # compute loss
            grads = tape.gradient(loss, model.trainable_variables)  # backprop
            model.optimizer.apply_gradients(zip(grads, model.trainable_variables))  # update weights
            epoch_losses.append(loss.numpy())

        mean_loss = np.mean(epoch_losses)
        loss_list.append(mean_loss)
        print(f"Epoch {epoch + 1} done")

        # --- Validation phase ---
        correct = 0
        for x_val, y_val in val_loader:
            logits = model(x_val, training=False)               # no dropout/BN update
            preds  = tf.argmax(logits, axis=1)
            correct += tf.reduce_sum(tf.cast(preds == y_val, tf.int32)).numpy()

        accuracy = correct / n_val
        accuracy_list.append(accuracy)

        # Save best weights
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_weights  = model.get_weights()

        if print_:
            current_lr = model.optimizer.learning_rate.numpy()
            print(f"Learning rate         : {current_lr:.6f}")
            print(f"Train loss (epoch {epoch+1}): {mean_loss:.4f}")
            print(f"Val accuracy (epoch {epoch+1}): {accuracy:.4f}")

    # Restore best weights (equivalent to model.load_state_dict(best_model_wts))
    model.set_weights(best_weights)
    return accuracy_list, loss_list, model

In [ ]:
# Track time and run training
start_datetime = datetime.now()
start_time     = time.time()

accuracy_list, loss_list, model = train_model(
    model, train_dataset, val_dataset, n_epochs, n_val
)

end_datetime = datetime.now()
elapsed_time = time.time() - start_time

print("Training completed.")
print(f"Start Time   : {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"End Time     : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Elapsed Time : {elapsed_time:.2f} seconds")

In [ ]:
# Save model weights  (equivalent to torch.save(model.state_dict(), 'model.pt'))
model.save_weights('model_tf.weights.h5')
print("Model weights saved to model_tf.weights.h5")

In [ ]:
# Plot training loss and validation accuracy
plot_stuff(loss_list, accuracy_list)

---
## 4. Practice Exercise — Test the Model on a New Image

In [ ]:
# Download test images (same as original lab)
!wget -q "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/eTwtaXXHQQkxDlkFWHqRNw/test.jpg"
!wget -q "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/xGq5q1hvl7HQLhmVGAOfzQ/test1.jpg"

In [ ]:
# Rebuild the model architecture and load saved weights
base_model_inf = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model_inf.trainable = False

inputs_inf  = tf.keras.Input(shape=(224, 224, 3))
x_inf       = base_model_inf(inputs_inf, training=False)
x_inf       = layers.GlobalAveragePooling2D()(x_inf)
outputs_inf = layers.Dense(n_classes)(x_inf)
model_inf   = Model(inputs_inf, outputs_inf)

# Load trained weights
model_inf.load_weights('model_tf.weights.h5')
print("Model weights loaded.")

In [ ]:
def predict_image(image_path, model, class_names):
    """
    Loads an image, preprocesses it, runs inference, and displays the result.
    
    Parameters:
        image_path  : path to the image file
        model       : compiled Keras model
        class_names : list of class label strings
    """
    # Load and resize
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize((224, 224))

    # Convert to numpy array and add batch dimension
    img_array  = np.array(image_resized, dtype=np.float32)
    img_array  = preprocess_input(img_array)      # same normalization used during training
    img_tensor = np.expand_dims(img_array, axis=0)  # shape: (1, 224, 224, 3)

    # Inference
    logits          = model(img_tensor, training=False)
    predicted_class = tf.argmax(logits, axis=1).numpy()[0]

    print(f"The image was classified as: {class_names[predicted_class]}")
    plt.imshow(image)
    plt.title(f"Predicted: {class_names[predicted_class]}")
    plt.axis("off")
    plt.show()

# Test on both downloaded images
predict_image("test.jpg",  model_inf, class_names)
predict_image("test1.jpg", model_inf, class_names)

---
## Congratulations!

You've completed the TensorFlow version of the image classification lab using transfer learning.

### PyTorch → TensorFlow mapping summary

| PyTorch | TensorFlow/Keras |
|---|---|
| `models.resnet18(pretrained=True)` | `ResNet50(weights='imagenet', include_top=False)` |
| `param.requires_grad = False` | `base_model.trainable = False` |
| `nn.Linear(512, n)` | `GlobalAveragePooling2D()` + `Dense(n)` |
| `ImageFolder(...)` | `image_dataset_from_directory(...)` |
| `transforms.Compose(...)` | `preprocess_input` + `RandomFlip/RandomRotation` layers |
| `nn.CrossEntropyLoss()` | `SparseCategoricalCrossentropy(from_logits=True)` |
| `torch.optim.SGD(...)` | `SGD(learning_rate=lr, momentum=momentum)` |
| `CyclicLR(triangular2)` | Custom `CyclicLRCallback` |
| `model.eval()` + `torch.no_grad()` | `model(x, training=False)` |
| `torch.save(model.state_dict(), ...)` | `model.save_weights(...)` |
| `model.load_state_dict(...)` | `model.set_weights(...)` / `model.load_weights(...)` |